라이브러리 불러오기

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import re
from sklearn.linear_model import LinearRegression

RandomForest

In [3]:
df = pd.read_csv('../data/processed/iphone_resale_market_preprocessed.csv')
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.head()

(2176, 16)
['title', 'model_family', 'generation_number', 'is_pro', 'storage_gb', 'condition', 'condition_score', 'price', 'wasPrice', 'price_discount_pct', 'available', 'sold', 'seller', 'itemLocation', 'us_state', 'lastUpdated']
title                     str
model_family              str
generation_number     float64
is_pro                   bool
storage_gb              int64
condition                 str
condition_score         int64
price                 float64
wasPrice              float64
price_discount_pct    float64
available             float64
sold                  float64
seller                    str
itemLocation              str
us_state                  str
lastUpdated               str
dtype: object


,title,model_family,generation_number,is_pro,storage_gb,condition,condition_score,price,wasPrice,price_discount_pct,available,sold,seller,itemLocation,us_state,lastUpdated
0,Apple iPhone 14 128gb RED color (Factory Unloc...,iPhone 14,14.0,False,128,Used,2,329.99,NaN,NaN,NaN,NaN,Seller 019,"Granada Hills, California, United States",CA,2026-01-02
1,Lot of 10 Apple iPhone 12 Pro 128GB Unlocked M...,iPhone 12 Pro,12.0,True,128,Used,2,3160.00,NaN,NaN,NaN,NaN,Seller 332,"Winter Park, Florida, United States",FL,2026-03-01
2,Apple iPhone 14 Pro MAX 128GB FULLY Unlocked,iPhone 14 Pro Max,14.0,True,128,Used,2,726.99,NaN,NaN,3.0,NaN,Seller 019,"Granada Hills, California, United States",CA,2025-12-12
3,Apple iPhone 14 Pro MAX 256gb Space black (Fac...,iPhone 14 Pro Max,14.0,True,256,Used,2,669.99,NaN,NaN,NaN,NaN,Seller 019,"Granada Hills, California, United States",CA,2025-12-12
4,Apple iPhone 14 Pro MAX 512gb Deep purple (Fac...,iPhone 14 Pro Max,14.0,True,512,Used,2,659.99,NaN,NaN,2.0,NaN,Seller 019,"Granada Hills, California, United States",CA,2026-03-10


Train/Test 분리 + 전처리 확인

In [4]:
# [c for c in df.columns if "price" in c.lower()]

In [5]:
# price_cols = [c for c in df.columns if "price" in c.lower()]
X = df.drop(columns=["price"])
y = df["price"]

print(X.shape, y.shape)
X.head()

(2176, 15) (2176,)


,title,model_family,generation_number,is_pro,storage_gb,condition,condition_score,wasPrice,price_discount_pct,available,sold,seller,itemLocation,us_state,lastUpdated
0,Apple iPhone 14 128gb RED color (Factory Unloc...,iPhone 14,14.0,False,128,Used,2,NaN,NaN,NaN,NaN,Seller 019,"Granada Hills, California, United States",CA,2026-01-02
1,Lot of 10 Apple iPhone 12 Pro 128GB Unlocked M...,iPhone 12 Pro,12.0,True,128,Used,2,NaN,NaN,NaN,NaN,Seller 332,"Winter Park, Florida, United States",FL,2026-03-01
2,Apple iPhone 14 Pro MAX 128GB FULLY Unlocked,iPhone 14 Pro Max,14.0,True,128,Used,2,NaN,NaN,3.0,NaN,Seller 019,"Granada Hills, California, United States",CA,2025-12-12
3,Apple iPhone 14 Pro MAX 256gb Space black (Fac...,iPhone 14 Pro Max,14.0,True,256,Used,2,NaN,NaN,NaN,NaN,Seller 019,"Granada Hills, California, United States",CA,2025-12-12
4,Apple iPhone 14 Pro MAX 512gb Deep purple (Fac...,iPhone 14 Pro Max,14.0,True,512,Used,2,NaN,NaN,2.0,NaN,Seller 019,"Granada Hills, California, United States",CA,2026-03-10


In [6]:
# 범주형 데이터 원핫인코딩 필요
cat_cols = X.select_dtypes(include="object").columns.tolist()
print("범주형 컬럼:", cat_cols)
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(X.shape)
X.head()

범주형 컬럼: ['title', 'model_family', 'condition', 'seller', 'itemLocation', 'us_state', 'lastUpdated']
(2176, 3272)


C:\Users\EZ\AppData\Local\Temp\ipykernel_18004\329843962.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns.tolist()


,generation_number,is_pro,storage_gb,condition_score,wasPrice,price_discount_pct,available,sold,title_(New) Apple iPhone 17 5G 8GB+256GB SAGE Dual SIM Unlocked iOS Cell Phone,title_*NEW OEM SEALED* Apple iPhone 16 - 128GB - Black (Unlocked)Apple 1yr Warranty,...,lastUpdated_2026-03-20,lastUpdated_2026-03-21,lastUpdated_2026-03-22,lastUpdated_2026-03-23,lastUpdated_2026-03-24,lastUpdated_2026-03-25,lastUpdated_2026-03-26,lastUpdated_2026-03-27,lastUpdated_2026-03-28,lastUpdated_2026-03-29
0,14.0,False,128,2,NaN,NaN,NaN,NaN,False,False,...,False,False,False,False,False,False,False,False,False,False
1,12.0,True,128,2,NaN,NaN,NaN,NaN,False,False,...,False,False,False,False,False,False,False,False,False,False
2,14.0,True,128,2,NaN,NaN,3.0,NaN,False,False,...,False,False,False,False,False,False,False,False,False,False
3,14.0,True,256,2,NaN,NaN,NaN,NaN,False,False,...,False,False,False,False,False,False,False,False,False,False
4,14.0,True,512,2,NaN,NaN,2.0,NaN,False,False,...,False,False,False,False,False,False,False,False,False,False


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(1740, 3272) (436, 3272)


In [8]:
# train_test_split 직후, 모델 학습 전에 한번 확인
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

wasPrice              1640
price_discount_pct    1640
available              836
sold                  1072
dtype: int64


RandomForestRegressor 학습 + 평가 지표

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

preds = rf_model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
mse = mean_squared_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("===== RandomForest =====")
print(f"RF_MAE: {mae}")
print(f"RF_MSE: {mse}")
print(f"RF_RMSE: {rmse}")
print(f"RF_R2: {r2}")

===== RandomForest =====
RF_MAE: 84.11207749927186
RF_MSE: 103570.71489188539
RF_RMSE: 321.8240433713513
RF_R2: 0.5502851837999576


----
XGBoost

In [10]:
xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
)

# ========================================================
# XGBoost가 허용하지 않는 특수문자를 언더스코어로 치환
X_train.columns = [re.sub(r"[\[\]<>]", "_", col) for col in X_train.columns]
X_test.columns = [re.sub(r"[\[\]<>]", "_", col) for col in X_test.columns]
# =========================================================

xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_mse = mean_squared_error(y_test, xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_r2 = r2_score(y_test, xgb_preds)

print("===== XGBoost =====")
print(f"XGB_MAE: {xgb_mae}")
print(f"XGB_MSE: {xgb_mse}")
print(f"XGB_RMSE: {xgb_rmse}")
print(f"XGB_R2: {xgb_r2}")

===== XGBoost =====
XGB_MAE: 91.55183052482954
XGB_MSE: 113221.36081438539
XGB_RMSE: 336.48381954320683
XGB_R2: 0.5083810754641291


---
LinearRegression

In [11]:
X_train.isnull().sum()[X_train.isnull().sum() > 0]

wasPrice              1640
price_discount_pct    1640
available              836
sold                  1072
dtype: int64

In [12]:
# 학습 데이터 기준으로 결측치 채우기
X_train_filled = X_train.fillna(X_train.mean())
X_test_filled = X_test.fillna(X_train.mean())  # test도 train의 평균값 기준으로 채워야 함 (data leakage 방지)

In [13]:
lr_model = LinearRegression()
lr_model.fit(X_train_filled, y_train)

lr_preds = lr_model.predict(X_test_filled)

lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)

print("===== LinearRegression =====")
print(f"LR_MAE: {lr_mae}")
print(f"LR_RMSE: {lr_rmse}")
print(f"LR_R2: {lr_r2}")

===== LinearRegression =====
LR_MAE: 108.12981582806069
LR_RMSE: 318.55693295791536
LR_R2: 0.5593697136413265


In [14]:
# 확인해보기
print(df["model_family"].nunique())
print(df["generation_number"].nunique())

20
6


In [15]:
print(df.shape)

(2176, 16)


In [ ]:
# 최종 모델 선택 (여기선 RandomForest 예시)
final_model = rf_model
final_preds = preds  # rf_model의 predict 결과

# 가격 범위 계산에 쓸 residual 표준편차
residuals = y_test - final_preds
residual_std = np.std(residuals)
print(f"residual_std: {residual_std}")

# predict.py에서 인코딩을 똑같이 맞추기 위해 학습 때 쓴 컬럼 목록도 저장
feature_columns = X_train.columns.tolist()

residual_std: 321.25


In [17]:
import joblib

joblib.dump({
    "model": final_model,
    "feature_columns": feature_columns,
    "residual_std": residual_std,
}, "../models/price_model.pkl")

print("저장 완료: models/price_model.pkl")

저장 완료: models/price_model.pkl
